# OpTC Host ML — Isolation Forest, random row subsample (shared shards)

This notebook reads the same persisted 75/10/15 split and vocabulary as the other models.
Validation and test retain every event, exactly like the other notebooks.

Isolation Forest does not use labels to fit, and each of its trees already trains on only
`max_samples` rows drawn from whatever array it's given — it was designed for exactly this
"billions of mostly-normal events, a vanishingly rare anomaly class" shape of problem, and does
not need anywhere near the full 3.48B training rows to build useful trees. We still need *some*
array resident in memory to draw those per-tree bootstraps from, so this notebook takes one
uniform random subsample of the train stream (no label stratification — the positive rate in the
subsample is left as in the population) sized well below the RAM budget.


Trade-off versus the supervised models (rf/lgbm/xgb) in this project: Isolation Forest never sees
the rare-event label during training, only during threshold selection/evaluation, so it is
scoring general outlier-ness rather than the specific labeled attack pattern. Expect it to be a
useful complementary signal rather than a drop-in replacement for the supervised trees. It is also
the cheapest of the four models to train — worth running first as a sanity check on the pipeline.


In [ ]:
from pathlib import Path
import sys, json
# Set an explicit path here only if several Host input datasets are attached.
MANIFEST = Path(
    '/kaggle/input/datasets/nhatnguyen2101/'
    'optc-host-ml-full-input/manifest.json'
)
if MANIFEST is None:
    candidates = []
    for path in Path('/kaggle/input').rglob('manifest.json'):
        try:
            meta = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if isinstance(meta, dict) and meta.get('format') == 'host_binary_delta_npz':
            candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one Host dataset; set MANIFEST explicitly. Found: {candidates}')
    MANIFEST = candidates[0]
MANIFEST = Path(MANIFEST)
meta = json.loads(MANIFEST.read_text())
VOLUME_ROOTS = {key: MANIFEST.parent / 'volumes' / key for key in meta['volumes']}
sys.path.insert(0, str(MANIFEST.parent))
from host_shards import HostShardDataset
from shard_metrics import evaluate_stream
data = HostShardDataset(MANIFEST, VOLUME_ROOTS)
print('Manifest:', MANIFEST)
# Decode one shard per split to check the uploaded paths and format.
for split in ('train', 'val', 'test'):
    X_check, y_check = next(data.iter_batches(split))
    print(split, 'first shard:', X_check.shape, 'positive labels:', int(y_check.sum()))
    del X_check, y_check
print(json.dumps(data.meta['splits'], indent=2))
print('Full train CSR lower bound GiB:', data.csr_bytes('train') / 2**30)
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)


In [ ]:
MODEL_NAME = 'isoforest'
import pickle
import numpy as np
from scipy import sparse
from sklearn.ensemble import IsolationForest

# Uniform random subsample of the training stream, streamed shard-by-shard so
# peak memory stays at one shard. No label stratification: this is a plain
# random sample of rows, matching the natural (extremely skewed) class ratio.
TRAIN_SAMPLE_ROWS = 4_000_000
SEED = 1
rng = np.random.default_rng(SEED)

train_rows = data.meta['splits']['train']['rows']
keep_p = min(1.0, TRAIN_SAMPLE_ROWS / train_rows)
print(f'uniform row keep-probability: {keep_p:.6f}')

X_parts, y_parts = [], []
n_kept = 0
for X_shard, y_shard in data.iter_batches('train'):
    keep = rng.random(X_shard.shape[0]) < keep_p
    if keep.any():
        X_parts.append(X_shard[keep]); y_parts.append(y_shard[keep])
        n_kept += int(keep.sum())

X_train = sparse.vstack(X_parts, format='csr')
y_train = np.concatenate(y_parts)
del X_parts
print(f'sampled {n_kept} rows ({int(y_train.sum())} positives) -> X_train {X_train.shape}')

# contamination='auto' sidesteps a known sklearn bug combining sparse input
# with a float contamination value; we pick our own operating threshold from
# the validation stream below anyway, so the built-in decision boundary from
# `contamination` is not used.
model = IsolationForest(n_estimators=300, max_samples=256,
    contamination='auto', random_state=SEED, n_jobs=4)
model.fit(X_train)

# score_samples: higher = more normal, lower = more anomalous. shard_metrics
# requires predict(X) in [0,1], so rescale monotonically using percentiles of
# the fitted sample's own score distribution (1st/99th to resist outliers).
# This only relabels the score axis; it does not change the ranking that the
# F1-threshold search and binned AUC/PR metrics depend on.
ref_scores = model.score_samples(X_train)
lo, hi = np.percentile(ref_scores, [1, 99])
print(f'reference anomaly-score range (1st/99th pct): {lo:.4f} / {hi:.4f}')

def predict(X):
    raw = model.score_samples(X)
    p = (hi - raw) / max(hi - lo, 1e-12)
    return np.clip(p, 0.0, 1.0)

del X_train, y_train
with (OUTPUT / 'isoforest_model.pkl').open('wb') as handle:
    pickle.dump(model, handle)


In [ ]:
validation = evaluate_stream(data, 'val', predict)
test = evaluate_stream(data, 'test', predict, threshold=validation['threshold'])
report = {'model': MODEL_NAME, 'split_sha256':data.meta['split_sha256'],
    'vocabulary_sha256':data.meta['vocabulary_sha256'], 'seed':SEED,
    'train_sampling': {'method':'uniform_row_subsample', 'label_stratified': False,
        'rows_sampled':n_kept, 'rows_target':TRAIN_SAMPLE_ROWS,
        'score_rescale_percentiles':{'lo_raw':float(lo), 'hi_raw':float(hi)}},
    'validation':validation, 'test':test}
(OUTPUT / (MODEL_NAME + '_metrics.json')).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
